In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os

#### Spark session setup

In [3]:
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = (
    SparkSession.builder
    .appName("raw_to_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

:: loading settings :: url = jar:file:/Users/nikos/Desktop/zrh-curfew-predictor/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nikos/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nikos/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9c413149-e0ae-4c58-917d-3353f9debd35;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	fou

In [ ]:
# Read the raw arrivals data from JSON files
arrivals_raw = spark.read.option("multiLine", True).json("../data/raw/arrivals/*.json")

arrivals_raw.printSchema()
print("Rows:", arrivals_raw.count())
arrivals_raw.show(5)


26/09/14 12:27:26 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/arrivals/*.json.
java.io.FileNotFoundException: File ../data/raw/arrivals/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

root
 |-- arrivalAirportCandidatesCount: long (nullable = true)
 |-- callsign: string (nullable = true)
 |-- departureAirportCandidatesCount: long (nullable = true)
 |-- estArrivalAirport: string (nullable = true)
 |-- estArrivalAirportHorizDistance: long (nullable = true)
 |-- estArrivalAirportVertDistance: long (nullable = true)
 |-- estDepartureAirport: string (nullable = true)
 |-- estDepartureAirportHorizDistance: long (nullable = true)
 |-- estDepartureAirportVertDistance: long (nullable = true)
 |-- firstSeen: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lastSeen: long (nullable = true)

Rows: 12543
+-----------------------------+--------+-------------------------------+-----------------+------------------------------+-----------------------------+-------------------+--------------------------------+-------------------------------+----------+------+----------+
|arrivalAirportCandidatesCount|callsign|departureAirportCandidatesCount|estArrivalAirport|estArriva

In [ ]:
# Write to bronze layer
from pyspark.sql import functions as F

arrivals_bronze = (
    arrivals_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
)

(
    arrivals_bronze.write
    .format("delta")
    .mode("overwrite")    
    .save("../data/bronze/arrivals")
)

26/09/14 12:39:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/14 12:39:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
# Repeat the same process for departures data

departures_raw = spark.read.option("multiLine", True).json("../data/raw/departures/*.json")

departures_raw.printSchema()
print("Rows:", departures_raw.count())
departures_raw.show(5)

26/09/14 12:47:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/departures/*.json.
java.io.FileNotFoundException: File ../data/raw/departures/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catal

root
 |-- arrivalAirportCandidatesCount: long (nullable = true)
 |-- callsign: string (nullable = true)
 |-- departureAirportCandidatesCount: long (nullable = true)
 |-- estArrivalAirport: string (nullable = true)
 |-- estArrivalAirportHorizDistance: long (nullable = true)
 |-- estArrivalAirportVertDistance: long (nullable = true)
 |-- estDepartureAirport: string (nullable = true)
 |-- estDepartureAirportHorizDistance: long (nullable = true)
 |-- estDepartureAirportVertDistance: long (nullable = true)
 |-- firstSeen: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lastSeen: long (nullable = true)

Rows: 12512
+-----------------------------+--------+-------------------------------+-----------------+------------------------------+-----------------------------+-------------------+--------------------------------+-------------------------------+----------+------+----------+
|arrivalAirportCandidatesCount|callsign|departureAirportCandidatesCount|estArrivalAirport|estArriva

In [10]:
departures_bronze = (
    departures_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
)

(
    departures_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .save("../data/bronze/departures")
)

#### Reference data

In [9]:

aircraft_raw = (
    spark.read
    .option("header", True)
    .option("quote", "'")
    .csv("../data/raw/reference/aircraft-database-complete-2025-08.csv")
)


print("Rows:", aircraft_raw.count())
aircraft_raw.printSchema()
aircraft_raw.select("icao24", "registration", "typecode", "model", "operator").show(10, truncate=False)


Rows: 616743
root
 |-- icao24: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- acars: string (nullable = true)
 |-- adsb: string (nullable = true)
 |-- built: string (nullable = true)
 |-- categoryDescription: string (nullable = true)
 |-- country: string (nullable = true)
 |-- engines: string (nullable = true)
 |-- firstFlightDate: string (nullable = true)
 |-- firstSeen: string (nullable = true)
 |-- icaoAircraftClass: string (nullable = true)
 |-- lineNumber: string (nullable = true)
 |-- manufacturerIcao: string (nullable = true)
 |-- manufacturerName: string (nullable = true)
 |-- model: string (nullable = true)
 |-- modes: string (nullable = true)
 |-- nextReg: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- operatorCallsign: string (nullable = true)
 |-- operatorIata: string (nullable = true)
 |-- operatorIcao: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- prevReg: string (null

In [11]:
# Write to bronze layer
aircraft_bronze = (
    aircraft_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
)

(
    aircraft_bronze
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .save("../data/bronze/aircraft")
)